# Notebook 30. IMERG precipitation and 925-hPa convergence

This notebook tests whether stronger low-level convergence during catalogued JPCZ episodes is associated with greater **GPM IMERG Final V07** precipitation. It replaces the earlier moisture-proxy comparison with NASA's satellite–gauge precipitation estimate.

## Analysis choices

- **IMERG field:** `Grid/precipitationCal`, the complete gauge-calibrated precipitation rate (mm h$^{-1}$), at half-hourly 0.1° resolution.
- **Convergence:** the 12-hour trailing mean of area-weighted 925-hPa divergence ending at each event peak, recomputed from ERA5 wind with the same MetPy divergence method used by the detector. The plotted predictor is $-D$, so larger positive values mean stronger convergence.
- **Regions:** the digitized JPCZ polygon and the editable Sea-of-Japan coastal wedge from Notebook 22.
- **Accumulation:** area-weighted mean precipitation depth (mm) across the detector episode, expanded 11 hours before its first threshold label because a 12-hour trailing $D$ window produced that label.
- **Mean rate:** the event accumulation divided by that event window's duration (mm h$^{-1}$). This is the primary comparison because total accumulation also grows when an event lasts longer.
- **Inference:** Pearson correlation and two-sided simple linear regression, with 95% confidence intervals. The stated null is no linear association ($r=0$, regression slope $=0$).

The notebook does **not** change the catalog. It automatically uses the current merged catalog in Google Drive (`jpcz_catalog_ndjf_merged_12h.csv`) when that file exists; `CATALOG_OVERRIDE_PATH` is available only if a different catalog is intentionally selected. IMERG begins in June 2000, so earlier catalog events are explicitly excluded.

All derived files are mirrored to Google Drive after every completed day/event batch. Original global IMERG HDF5 granules stay in temporary Colab storage by default; Drive stores the compact regional time series and analysis outputs needed to resume.

In [ ]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

REPO_URL = 'https://github.com/angelicasophyaramirez-blip/JPCZcatalogcolab.git'
BRANCH = os.environ.get('JPCZ_CATALOG_BRANCH', 'codex/notebook16-pcolormesh')
REPO_DIR = '/content/JPCZcatalog'
FORCE_REFRESH_REPO = False

PERSIST_OUTPUTS_TO_DRIVE = True
DRIVE_ROOT = Path('/content/drive/MyDrive/JPCZcatalog_outputs')

if PERSIST_OUTPUTS_TO_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT.mkdir(parents=True, exist_ok=True)

def clone_repo_branch():
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, REPO_DIR], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)

if FORCE_REFRESH_REPO and os.path.exists(REPO_DIR):
    shutil.rmtree(REPO_DIR)
if not os.path.exists(REPO_DIR):
    clone_repo_branch()
else:
    subprocess.run(['git', '-C', REPO_DIR, 'fetch', 'origin', BRANCH], check=True)
    subprocess.run(['git', '-C', REPO_DIR, 'checkout', '-B', BRANCH, f'origin/{BRANCH}'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', f'{REPO_DIR}/requirements-colab.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', REPO_DIR], check=True)

os.chdir(REPO_DIR)
if f'{REPO_DIR}/src' not in sys.path:
    sys.path.insert(0, f'{REPO_DIR}/src')
print('Repository:', REPO_DIR)
print('Branch:', subprocess.run(['git', '-C', REPO_DIR, 'branch', '--show-current'], text=True, capture_output=True, check=True).stdout.strip())
print('Drive checkpoint root:', DRIVE_ROOT)

In [ ]:
import warnings

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

from jpcz_catalog.config import BoundingBox, JPCZ_POLYGON_VERTICES
from jpcz_catalog.detect import compute_divergence_stack, prepare_detection_geometry
from jpcz_catalog.era5 import open_arco_era5, subset_era5_window
from jpcz_catalog.imerg import (
    parse_imerg_granule_start,
    read_precipitation_cal_subset,
    region_mean_rates,
)

# ----- User controls -----
# Leave as None to use the current expanded merged catalog in Google Drive.
# Set a path only when deliberately testing a different catalog.
CATALOG_OVERRIDE_PATH = None

# Keep these False while reviewing the setup. Turn on one calculation at a time.
RUN_ERA5_REGIONAL_DIVERGENCE = False
RUN_IMERG_DOWNLOAD = False
MAX_IMERG_DAYS_THIS_RUN = 3  # None processes every still-missing event day.
DELETE_LOCAL_GRANULES_AFTER_CHECKPOINT = False

# IMERG became available on 2000-06-01. Require 90% of expected half-hours for an event metric.
IMERG_FIRST_VALID_TIME = pd.Timestamp('2000-06-01 00:00:00')
MIN_TEMPORAL_COVERAGE = 0.90

# Same editable coastal region used in Notebook 22. It includes overlap with the JPCZ polygon.
COASTAL_WEDGE_VERTICES = (
    (133.05, 35.55),
    (136.05, 35.55),
    (139.55, 39.00),
    (139.55, 42.55),
)
REGIONS = {
    'jpcz_polygon': JPCZ_POLYGON_VERTICES,
    'coastal_wedge': COASTAL_WEDGE_VERTICES,
}

# One-cell halo keeps the regional masks away from the numerical outer boundary.
ERA5_DIVERGENCE_DOMAIN = BoundingBox(lon_min=128.0, lon_max=141.0, lat_min=35.0, lat_max=43.0)
IMERG_READ_DOMAIN = BoundingBox(lon_min=128.0, lon_max=141.0, lat_min=35.0, lat_max=43.0)

ANALYSIS_DIR = Path('outputs/verification/imerg_precipitation_convergence')
ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)
DRIVE_ANALYSIS_DIR = DRIVE_ROOT / 'imerg_precipitation_convergence'
DRIVE_ANALYSIS_DIR.mkdir(parents=True, exist_ok=True)

ERA5_HOURLY_PATH = DRIVE_ANALYSIS_DIR / 'era5_925hpa_regional_hourly_divergence.csv'
IMERG_RATE_PATH = DRIVE_ANALYSIS_DIR / 'imerg_regional_halfhourly_rates.csv'
EVENT_METRICS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_event_precipitation_metrics.csv'
STATS_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_statistics.csv'
PLOT_PATH = DRIVE_ANALYSIS_DIR / 'imerg_convergence_association_scatter.png'
CATALOG_MANIFEST_PATH = DRIVE_ANALYSIS_DIR / 'imerg_catalog_run_manifest.csv'
LOCAL_GRANULE_DIR = Path('/content/imerg_granules')

def mirror_to_repo(path):
    destination = ANALYSIS_DIR / path.name
    shutil.copy2(path, destination)

def catalog_candidates():
    return [
        Path(CATALOG_OVERRIDE_PATH) if CATALOG_OVERRIDE_PATH else None,
        DRIVE_ROOT / 'jpcz_catalog_ndjf_merged_12h.csv',
        Path('outputs/verification/jpcz_catalog_ndjf_merged_12h.csv'),
    ]

catalog_path = next((path for path in catalog_candidates() if path is not None and path.exists()), None)
if catalog_path is None:
    raise FileNotFoundError('No merged catalog was found. Set CATALOG_OVERRIDE_PATH or add the merged catalog to Drive.')
print('Catalog selected:', catalog_path)

In [ ]:
catalog = pd.read_csv(catalog_path)
for column in ('event_start', 'event_end', 'event_peak'):
    catalog[column] = pd.to_datetime(catalog[column])

catalog = catalog.sort_values('event_start').reset_index(drop=True)
catalog['event_id'] = catalog['event_peak'].dt.strftime('%Y%m%dT%H%M')
catalog['analysis_start'] = catalog['event_start'] - pd.Timedelta(hours=11)
catalog['analysis_end_exclusive'] = catalog['event_end'] + pd.Timedelta(hours=1)
catalog['analysis_window_hours'] = (catalog['analysis_end_exclusive'] - catalog['analysis_start']).dt.total_seconds() / 3600

pre_imerg = catalog['analysis_start'] < IMERG_FIRST_VALID_TIME
catalog_for_imerg = catalog.loc[~pre_imerg].copy()
print(f'Merged catalog rows: {len(catalog)}')
print(f'Excluded before IMERG coverage ({IMERG_FIRST_VALID_TIME:%Y-%m-%d}): {pre_imerg.sum()}')
print(f'Eligible JPCZ episodes: {len(catalog_for_imerg)}')
print('Catalog peak-date coverage:', catalog.event_peak.min(), 'to', catalog.event_peak.max())
if catalog.event_peak.max().year <= 2018:
    warnings.warn('The selected catalog ends in 2018 or earlier. Confirm that Notebook 06 has saved the expanded merged CSV to Google Drive.')

catalog_manifest = pd.DataFrame([{
    'catalog_source': str(catalog_path),
    'catalog_rows': len(catalog),
    'imerg_eligible_rows': len(catalog_for_imerg),
    'excluded_before_imerg_rows': int(pre_imerg.sum()),
    'first_event_peak_utc': catalog['event_peak'].min(),
    'last_event_peak_utc': catalog['event_peak'].max(),
}])
catalog_manifest.to_csv(CATALOG_MANIFEST_PATH, index=False)
mirror_to_repo(CATALOG_MANIFEST_PATH)
print('Saved catalog run manifest:', CATALOG_MANIFEST_PATH)
display(catalog_for_imerg[['event_id', 'event_start', 'event_end', 'event_peak', 'duration_hours', 'analysis_window_hours']].head())

In [ ]:
def add_map_features(ax, title):
    ax.set_extent([127.5, 141.5, 34.5, 43.5], crs=ccrs.PlateCarree())
    ax.add_feature(cfeature.LAND, facecolor='#f3eee4', zorder=0)
    ax.coastlines(resolution='50m', linewidth=0.8)
    gridlines = ax.gridlines(draw_labels=True, linewidth=0.4, color='gray', alpha=0.65, linestyle='--')
    gridlines.top_labels = False
    gridlines.right_labels = False
    ax.set_title(title)

fig, ax = plt.subplots(figsize=(8.4, 6.2), subplot_kw={'projection': ccrs.PlateCarree()})
add_map_features(ax, 'IMERG precipitation regions and ERA5 convergence regions')
for name, vertices, color in [
    ('JPCZ polygon', JPCZ_POLYGON_VERTICES, '#1f78b4'),
    ('Coastal wedge', COASTAL_WEDGE_VERTICES, '#d95f02'),
]:
    lons, lats = zip(*(vertices + (vertices[0],)))
    ax.plot(lons, lats, color=color, linewidth=2.5, transform=ccrs.PlateCarree(), label=name)
ax.legend(loc='upper left')
fig.tight_layout()
region_plot_path = DRIVE_ANALYSIS_DIR / 'imerg_convergence_region_definitions.png'
fig.savefig(region_plot_path, dpi=180, bbox_inches='tight')
mirror_to_repo(region_plot_path)
plt.show()

## 1. Rebuild the regional ERA5 convergence signal

The detector's catalog column is the JPCZ-polygon $D$ value. For a fair four-panel regional comparison, this cell recomputes the same hourly 925-hPa divergence over both regions and applies a **trailing 12-hour mean** ending at each event peak. It caches the hourly regional series, so later runs only request peak windows that are missing.

Leave `RUN_ERA5_REGIONAL_DIVERGENCE = False` while reviewing the notebook. Set it to `True` to build/rebuild the checkpoint.

In [ ]:
def load_checkpoint(path, time_column):
    if not path.exists():
        return pd.DataFrame(columns=[time_column])
    frame = pd.read_csv(path)
    frame[time_column] = pd.to_datetime(frame[time_column])
    return frame

def weighted_mean(field, weights):
    valid_weights = weights.where(field.notnull())
    return (field * valid_weights).sum(dim=('latitude', 'longitude')) / valid_weights.sum(dim=('latitude', 'longitude'))

def required_era5_times(events):
    windows = [pd.date_range(peak - pd.Timedelta(hours=11), peak, freq='1h') for peak in events['event_peak']]
    return pd.DatetimeIndex(np.unique(np.concatenate([window.values for window in windows])))

era5_hourly = load_checkpoint(ERA5_HOURLY_PATH, 'time')
required_times = required_era5_times(catalog_for_imerg)
existing_times = pd.DatetimeIndex(era5_hourly['time']) if not era5_hourly.empty else pd.DatetimeIndex([])
missing_times = required_times.difference(existing_times)
print(f'ERA5 regional hourly checkpoint: {len(existing_times)} rows; {len(missing_times)} requested hours still missing.')

if RUN_ERA5_REGIONAL_DIVERGENCE and len(missing_times):
    arco = open_arco_era5(chunks={})
    for period in pd.PeriodIndex(missing_times, freq='M').unique():
        month_times = missing_times[pd.PeriodIndex(missing_times, freq='M') == period]
        month = subset_era5_window(
            arco,
            str(month_times.min()),
            str(month_times.max()),
            domain=ERA5_DIVERGENCE_DOMAIN,
            variables=('u_component_of_wind', 'v_component_of_wind'),
            level=925,
        ).sel(time=month_times).load()
        geometries = {name: prepare_detection_geometry(month.longitude, month.latitude, vertices) for name, vertices in REGIONS.items()}
        divergence = compute_divergence_stack(month, dx=geometries['jpcz_polygon'].dx, dy=geometries['jpcz_polygon'].dy)
        row = {'time': pd.to_datetime(divergence.time.values)}
        for name, geometry in geometries.items():
            row[f'{name}_divergence_s-1'] = weighted_mean(divergence, geometry.weights).values
        era5_hourly = pd.concat([era5_hourly, pd.DataFrame(row)], ignore_index=True).drop_duplicates('time').sort_values('time')
        era5_hourly.to_csv(ERA5_HOURLY_PATH, index=False)
        mirror_to_repo(ERA5_HOURLY_PATH)
        print(f'Completed and checkpointed ERA5: {period} ({len(era5_hourly)} hourly rows saved)')

def event_convergence_metrics(events, hourly):
    if hourly.empty:
        return pd.DataFrame({'event_id': events['event_id']})
    indexed = hourly.set_index('time').sort_index()
    rows = []
    for event in events.itertuples(index=False):
        expected = pd.date_range(event.event_peak - pd.Timedelta(hours=11), event.event_peak, freq='1h')
        window = indexed.reindex(expected)
        row = {'event_id': event.event_id, 'convergence_valid_hours': int(window.notna().all(axis=1).sum())}
        for name in REGIONS:
            d_column = f'{name}_divergence_s-1'
            d_12h = window[d_column].mean() if window[d_column].notna().sum() == 12 else np.nan
            row[f'{name}_D12_s-1'] = d_12h
            row[f'{name}_convergence_1e5_s-1'] = -d_12h * 1e5 if pd.notna(d_12h) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

event_convergence = event_convergence_metrics(catalog_for_imerg, era5_hourly)
display(event_convergence.head())

## 2. Download and reduce IMERG in resumable daily batches

NASA Earthdata Login is required on the first download. The code searches `GPM_3IMERGHH`, version `07`, obtains the needed **Final** half-hourly granules, reads only the Japan subdomain from each global file, and immediately checkpoints the two area-weighted regional rates to Drive. It does not save a second copy of the global raw files in Drive.

A small `MAX_IMERG_DAYS_THIS_RUN` is intentional: it lets you verify the first maps/tables and makes an interrupted Colab session cheap to resume. Set it to `None` once the first batch looks correct.

In [ ]:
def required_imerg_times(events):
    windows = [pd.date_range(row.analysis_start, row.analysis_end_exclusive, freq='30min', inclusive='left') for row in events.itertuples(index=False)]
    return pd.DatetimeIndex(np.unique(np.concatenate([window.values for window in windows])))

imerg_rates = load_checkpoint(IMERG_RATE_PATH, 'time')
required_halfhours = required_imerg_times(catalog_for_imerg)
existing_halfhours = pd.DatetimeIndex(imerg_rates['time']) if not imerg_rates.empty else pd.DatetimeIndex([])
missing_halfhours = required_halfhours.difference(existing_halfhours)
missing_days = pd.DatetimeIndex(missing_halfhours.normalize().unique()).sort_values()
print(f'IMERG regional half-hour checkpoint: {len(existing_halfhours)} rows; {len(missing_halfhours)} needed half-hours still missing across {len(missing_days)} UTC days.')

if RUN_IMERG_DOWNLOAD and len(missing_days):
    import earthaccess

    earthaccess.login()  # interactive NASA Earthdata Login if credentials are not already available
    day_limit = len(missing_days) if MAX_IMERG_DAYS_THIS_RUN is None else min(MAX_IMERG_DAYS_THIS_RUN, len(missing_days))
    new_rows = []
    for day in missing_days[:day_limit]:
        next_day = day + pd.Timedelta(days=1)
        wanted = set(missing_halfhours[(missing_halfhours >= day) & (missing_halfhours < next_day)])
        results = earthaccess.search_data(
            short_name='GPM_3IMERGHH',
            version='07',
            temporal=(day.isoformat(), next_day.isoformat()),
            count=100,
        )
        local_day_dir = LOCAL_GRANULE_DIR / day.strftime('%Y%m%d')
        local_day_dir.mkdir(parents=True, exist_ok=True)
        files = earthaccess.download(results, local_path=local_day_dir, threads=4)
        daily_rows = []
        for file_path in files:
            try:
                timestamp = pd.Timestamp(parse_imerg_granule_start(file_path))
            except ValueError:
                continue
            if timestamp not in wanted:
                continue
            rate_field = read_precipitation_cal_subset(file_path, domain=IMERG_READ_DOMAIN)
            daily_rows.append({'time': timestamp, **{f'{name}_rate_mm_hr': value for name, value in region_mean_rates(rate_field, REGIONS).items()}})
        if daily_rows:
            imerg_rates = pd.concat([imerg_rates, pd.DataFrame(daily_rows)], ignore_index=True).drop_duplicates('time').sort_values('time')
            imerg_rates.to_csv(IMERG_RATE_PATH, index=False)
            mirror_to_repo(IMERG_RATE_PATH)
        print(f'Completed {day:%Y-%m-%d}: {len(daily_rows)} required IMERG half-hours saved.')
        if DELETE_LOCAL_GRANULES_AFTER_CHECKPOINT:
            shutil.rmtree(local_day_dir)

imerg_rates = load_checkpoint(IMERG_RATE_PATH, 'time')
display(imerg_rates.head())

In [ ]:
def event_precipitation_metrics(events, rates):
    if rates.empty:
        return pd.DataFrame({'event_id': events['event_id']})
    indexed = rates.set_index('time').sort_index()
    rows = []
    for event in events.itertuples(index=False):
        expected = pd.date_range(event.analysis_start, event.analysis_end_exclusive, freq='30min', inclusive='left')
        window = indexed.reindex(expected)
        row = {
            'event_id': event.event_id,
            'imerg_expected_halfhours': len(expected),
            'imerg_window_hours': len(expected) * 0.5,
        }
        for name in REGIONS:
            column = f'{name}_rate_mm_hr'
            valid = window[column].notna().sum()
            coverage = valid / len(expected)
            accumulation = window[column].sum(skipna=True) * 0.5 if coverage >= MIN_TEMPORAL_COVERAGE else np.nan
            row[f'{name}_imerg_valid_halfhours'] = int(valid)
            row[f'{name}_imerg_coverage_fraction'] = coverage
            row[f'{name}_imerg_accumulation_mm'] = accumulation
            row[f'{name}_imerg_mean_rate_mm_hr'] = accumulation / (len(expected) * 0.5) if pd.notna(accumulation) else np.nan
        rows.append(row)
    return pd.DataFrame(rows)

event_precip = event_precipitation_metrics(catalog_for_imerg, imerg_rates)
analysis = catalog_for_imerg.merge(event_convergence, on='event_id', how='left').merge(event_precip, on='event_id', how='left')
analysis.to_csv(EVENT_METRICS_PATH, index=False)
mirror_to_repo(EVENT_METRICS_PATH)
required_metric_columns = [
    'jpcz_polygon_convergence_1e5_s-1', 'coastal_wedge_convergence_1e5_s-1',
    'jpcz_polygon_imerg_mean_rate_mm_hr', 'coastal_wedge_imerg_mean_rate_mm_hr',
]
missing_metric_columns = [column for column in required_metric_columns if column not in analysis.columns]
if missing_metric_columns:
    print('No complete association rows yet. Build the ERA5 regional-divergence checkpoint, then the IMERG rate checkpoint.')
    print('Missing metric columns:', missing_metric_columns)
else:
    complete_rows = analysis.dropna(subset=required_metric_columns).shape[0]
    print('Complete event rows with both regional precipitation and convergence:', complete_rows, 'of', len(analysis))
display(analysis.head())

## 3. Association test and presentation-ready figures

Each point is one merged catalog episode. The four panels compare the two regions and two precipitation measures. The rate panels are the primary physical test; the accumulation panels are retained because they are intuitive for a presentation, but their slope can partly reflect the fact that longer episodes have more time to accumulate precipitation.

A statistically significant positive slope (or positive $r$) means that episodes with stronger low-level convergence tend to have greater IMERG precipitation on average. It does **not** imply that convergence is the sole cause of precipitation.

In [ ]:
def association_statistics(frame, *, x_column, y_column, region, measure):
    sample = frame[[x_column, y_column]].dropna()
    x = sample[x_column].to_numpy(float)
    y = sample[y_column].to_numpy(float)
    n = len(sample)
    if n < 4:
        return {'region': region, 'precipitation_measure': measure, 'n': n}
    correlation = stats.pearsonr(x, y)
    regression = stats.linregress(x, y)
    alpha = 0.05
    fisher_z = np.arctanh(correlation.statistic)
    z_margin = stats.norm.ppf(1 - alpha / 2) / np.sqrt(n - 3)
    r_ci_low, r_ci_high = np.tanh([fisher_z - z_margin, fisher_z + z_margin])
    slope_margin = stats.t.ppf(1 - alpha / 2, n - 2) * regression.stderr
    return {
        'region': region,
        'precipitation_measure': measure,
        'n': n,
        'pearson_r': correlation.statistic,
        'r_95ci_low': r_ci_low,
        'r_95ci_high': r_ci_high,
        'r_two_sided_p': correlation.pvalue,
        'slope': regression.slope,
        'slope_95ci_low': regression.slope - slope_margin,
        'slope_95ci_high': regression.slope + slope_margin,
        'slope_two_sided_p': regression.pvalue,
        'intercept': regression.intercept,
        'r_squared': regression.rvalue ** 2,
        'null_decision_alpha_0.05': 'reject H0' if correlation.pvalue < 0.05 else 'fail to reject H0',
    }

specifications = []
for region, region_label in [('jpcz_polygon', 'JPCZ polygon'), ('coastal_wedge', 'Coastal wedge')]:
    x_column = f'{region}_convergence_1e5_s-1'
    specifications.extend([
        (region_label, 'IMERG accumulation (mm)', x_column, f'{region}_imerg_accumulation_mm'),
        (region_label, 'IMERG mean rate (mm h$^{-1}$)', x_column, f'{region}_imerg_mean_rate_mm_hr'),
    ])

statistics_table = pd.DataFrame([
    association_statistics(analysis, x_column=x, y_column=y, region=region, measure=measure)
    for region, measure, x, y in specifications
])
if not statistics_table.empty:
    statistics_table.to_csv(STATS_PATH, index=False)
    mirror_to_repo(STATS_PATH)
display(statistics_table.round(4))

In [ ]:
def plot_association(ax, frame, *, x_column, y_column, title, y_label, summary):
    sample = frame[[x_column, y_column, 'duration_hours']].dropna()
    if len(sample) < 4:
        ax.text(0.5, 0.5, 'Run the two data-checkpoint cells first.', ha='center', va='center', transform=ax.transAxes)
        ax.set_title(title)
        return None
    points = ax.scatter(sample[x_column], sample[y_column], c=sample['duration_hours'], cmap='viridis', s=36, alpha=0.8, edgecolor='white', linewidth=0.35)
    fitted = stats.linregress(sample[x_column], sample[y_column])
    xline = np.linspace(sample[x_column].min(), sample[x_column].max(), 100)
    ax.plot(xline, fitted.intercept + fitted.slope * xline, color='#c0392b', linewidth=2.0)
    ax.set_title(title)
    ax.set_xlabel('925-hPa convergence strength, $-D_{12}$ (10$^{-5}$ s$^{-1}$)')
    ax.set_ylabel(y_label)
    ax.grid(alpha=0.25)
    annotation = f"n={int(summary['n'])}\nr={summary['pearson_r']:.2f} ({summary['r_95ci_low']:.2f}, {summary['r_95ci_high']:.2f})\np={summary['r_two_sided_p']:.3g}; {summary['null_decision_alpha_0.05']}"
    ax.text(0.03, 0.97, annotation, va='top', ha='left', transform=ax.transAxes, fontsize=9, bbox={'facecolor': 'white', 'edgecolor': '0.7', 'alpha': 0.9})
    return points

if statistics_table.empty or statistics_table['n'].fillna(0).max() < 4:
    raise RuntimeError('No complete event-level data yet. Set RUN_ERA5_REGIONAL_DIVERGENCE and RUN_IMERG_DOWNLOAD to True, then rerun the checkpoint cells.')

fig, axes = plt.subplots(2, 2, figsize=(14, 10), constrained_layout=True)
last_points = None
for ax, (region, measure, x_column, y_column) in zip(axes.flat, specifications):
    summary = statistics_table.loc[(statistics_table['region'] == region) & (statistics_table['precipitation_measure'] == measure)].iloc[0]
    last_points = plot_association(ax, analysis, x_column=x_column, y_column=y_column, title=f'{region}: {measure}', y_label=measure, summary=summary)
if last_points is not None:
    colorbar = fig.colorbar(last_points, ax=axes, shrink=0.82, pad=0.02)
    colorbar.set_label('Merged event duration (h)')
fig.suptitle('IMERG Final V07 precipitation versus ERA5 925-hPa convergence', fontsize=15)
fig.savefig(PLOT_PATH, dpi=220, bbox_inches='tight')
mirror_to_repo(PLOT_PATH)
plt.show()

In [ ]:
# Optional presentation table: compact, readable values and explicit interpretation.
presentation_table = statistics_table.copy()
for column in ['pearson_r', 'r_95ci_low', 'r_95ci_high', 'r_two_sided_p', 'slope', 'slope_95ci_low', 'slope_95ci_high', 'slope_two_sided_p', 'r_squared']:
    if column in presentation_table:
        presentation_table[column] = presentation_table[column].round(3)
display(presentation_table)

print('Interpretation guardrail: emphasize the rate panels for the physical association. Accumulation is presentation-friendly but is partly a duration-sensitive outcome; the point colors make that visible.')
print('Drive outputs:', DRIVE_ANALYSIS_DIR)

## Methods wording for the presentation

For each merged JPCZ episode, we calculated regional mean IMERG Final V07 gauge-calibrated precipitation (`precipitationCal`; half-hourly 0.1° grid) over the digitized JPCZ polygon and coastal wedge using cosine-latitude area weights. We derived event precipitation accumulation (mm) by summing half-hourly rates × 0.5 h across the detector episode and event mean precipitation rate (mm h$^{-1}$) by dividing accumulation by the episode window length. Low-level convergence was the negative of the 12-hour trailing, area-weighted 925-hPa ERA5 divergence ending at each catalog peak, with larger positive values denoting stronger convergence. We evaluated the precipitation–convergence association using two-sided Pearson correlation and ordinary least-squares regression, reporting $r$, 95% confidence intervals, slope, $R^2$, and $p$ values at $\alpha=0.05$.

Caution: IMERG is a satellite–gauge precipitation estimate, not a station observation. The result tests whether the estimates covary consistently with convergence across JPCZ episodes; it does not by itself establish causation.